In [ ]:
import numpy as np

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

REGION_TO_ELECTRODES = {
    'Middle_Finger': ['E1', 'E2', 'E3'],
    'Hand': ['E4', 'E5', 'E6', 'E7'],
    'Forearm': ['E8', 'E9', 'E10', 'E11', 'E12', 'E13'],
    'Arm': ['E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20']
}
ELECTRODE_TO_REGION = {e: r for r, es in REGION_TO_ELECTRODES.items() for e in es}
REGION_TO_LABEL = {'Middle_Finger': 0, 'Hand': 1, 'Forearm': 2, 'Arm': 3}
LABEL_TO_REGION = {v: k for k, v in REGION_TO_LABEL.items()}

In [ ]:
from pathlib import Path

BIDS_ROOT = Path("../../data/BIDS-somatosensory/BIDS-somatosensory")
DERIVATIVES = BIDS_ROOT / "derivatives" / "fmriprep"
subjects = ["sub-p0002"]
session, task, space = "ses-01", "task-S1Map", "MNI152NLin2009cAsym"
n_runs, HRF_DELAY, WINDOW = 4, 6.0, 1

RESULTS_DIR = Path("results/4_Classes_ANN_ICA")
SHAP_DIR = RESULTS_DIR / "shap"
SHAP_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import json

bold_json = BIDS_ROOT / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-1_bold.json"
with open(bold_json, 'r', encoding='utf-8') as f:
    tr = float(json.load(f)["RepetitionTime"])
print(f"TR: {tr} s")

In [ ]:
from nilearn.image import load_img, index_img, new_img_like, resample_to_img
from nilearn.datasets import fetch_atlas_destrieux_2009
from nilearn.maskers import NiftiMasker

first_run_path = DERIVATIVES / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-1_space-{space}_desc-preproc_bold.nii"
first_run_img = load_img(str(first_run_path))
ref_img = index_img(first_run_img, 0)

atlas = fetch_atlas_destrieux_2009()
s1_indices = [i for i, lab in enumerate(atlas.labels) if 'L G_postcentral' in str(lab) and i != 0]
atlas_img = load_img(atlas.maps)
mask_data = np.isin(atlas_img.get_fdata(), s1_indices).astype('uint8')
s1_mask = new_img_like(atlas_img, mask_data)
s1_mask_resampled = resample_to_img(s1_mask, ref_img, interpolation='nearest')
masker = NiftiMasker(mask_img=s1_mask_resampled, standardize=False)
masker.fit(first_run_img)
print(f"Voxels in S1 mask: {int(np.sum(s1_mask_resampled.get_fdata() > 0))}")

In [ ]:
import pandas as pd

X_list, y_list = [], []
for run in range(1, n_runs + 1):
    events_path = BIDS_ROOT / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-{run}_events.tsv"
    df = pd.read_csv(events_path, sep='\t')
    df['run'] = run
    stim = df[~df['trial_type'].isin(['Baseline', 'Jitter'])].copy()
    stim['region'] = stim['trial_type'].map(ELECTRODE_TO_REGION)

    bold_path = DERIVATIVES / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-{run}_space-{space}_desc-preproc_bold.nii"
    bold_img = load_img(str(bold_path))
    for _, event in stim.iterrows():
        vol_indx = int(np.round((event["onset"] + HRF_DELAY) / tr))
        vol_indices = [vol_indx - WINDOW, vol_indx, vol_indx + WINDOW]
        if all(0 <= v < bold_img.shape[3] for v in vol_indices):
            vols = [masker.transform(index_img(bold_img, v)) for v in vol_indices]
            vols = [v[0] if len(v.shape) == 2 else v for v in vols]
            X_list.append(np.mean(vols, axis=0))
            y_list.append(event["region"])

X = np.array(X_list)
y_encoded = np.array([REGION_TO_LABEL[r] for r in y_list])
print(f"X: {X.shape}, y: {y_encoded.shape}")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler_all = StandardScaler()
X_scaled = scaler_all.fit_transform(X)

In [ ]:
import torch
from torch.nn import Module, Linear, ReLU, Sequential, Dropout, BatchNorm1d

class SomatotopicANN(Module):
    def __init__(self, input_size, hidden1, hidden2, num_classes, dropout_rate):
        super().__init__()
        layers = [Linear(input_size, hidden1), BatchNorm1d(hidden1), ReLU(), Dropout(dropout_rate),
                  Linear(hidden1, hidden2), BatchNorm1d(hidden2), ReLU(), Dropout(dropout_rate),
                  Linear(hidden2, num_classes)]
        self.network = Sequential(*layers)
    def forward(self, x):
        return self.network(x)

#use the best model -> model_h64-16_wd0.0001_dr0.2.pt
model = SomatotopicANN(X_scaled.shape[1], 64, 16, 4, 0.2)
model.load_state_dict(torch.load(RESULTS_DIR / "models" / "model_h64-16_wd0.0001_dr0.2.pt", weights_only=True))
model.eval()
print("Model loaded.")

In [ ]:
import shap

X_tensor = torch.FloatTensor(X_scaled)
background = X_tensor[:20]

explainer = shap.DeepExplainer(model, background)
shap_out = explainer.shap_values(X_tensor)

# Handle both SHAP API versions as one can fail, 4 arrays (160,50)
if isinstance(shap_out, list):
    shap_values = shap_out
else:
    shap_values = [shap_out[:, :, c] for c in range(4)]

print(f"Classes: {len(shap_values)}, shape per class: {shap_values[0].shape}")

In [ ]:
import matplotlib.pyplot as plt

feature_names = [f"PC{i+1}" for i in range(50)]

for c in range(4):
    fig, ax = plt.subplots(figsize=(8, 5))
    shap.summary_plot(shap_values[c], X_scaled, feature_names=feature_names,
                      plot_type="bar", show=False)
    plt.title(f"SHAP Feature Importance — {LABEL_TO_REGION[c]}")
    plt.tight_layout()
    plt.savefig(SHAP_DIR / f"shap_pca_{LABEL_TO_REGION[c]}.png", dpi=150)
    plt.show()
    plt.close()

In [ ]:
n_voxels = X.shape[1]
shap_voxel = np.zeros((4, n_voxels))

for c in range(4):
    shap_voxel[c] = np.mean(shap_values[c][y_encoded == c], axis=0) 

print(f"shap_voxel shape: {shap_voxel.shape}")

In [ ]:
from nilearn import plotting, datasets
from IPython.display import display
import matplotlib.pyplot as plt
from nilearn import surface as surf_utils

FIGURES_DIR = SHAP_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

for c in range(4):
    class_name = LABEL_TO_REGION[c]
    shap_img = masker.inverse_transform(shap_voxel[c].reshape(1, -1))
    nonzero = shap_voxel[c][np.abs(shap_voxel[c]) > 0]
    thresh = np.percentile(np.abs(nonzero), 10) if len(nonzero) > 0 else 0.0

    view = plotting.view_img_on_surf(
        shap_img,
        surf_mesh='fsaverage',
        title=f'SHAP: {class_name}',
        symmetric_cmap=None,
        threshold=thresh,
        vol_to_surf_kwargs={"interpolation": "nearest_most_frequent"}
    )
    view.save_as_html(FIGURES_DIR / f"shap_surf_{class_name}.html")
    display(view)



In [ ]:
fsaverage = datasets.fetch_surf_fsaverage()

for c in range(4):
    class_name = LABEL_TO_REGION[c]
    shap_img = masker.inverse_transform(shap_voxel[c].reshape(1, -1))
    texture = surf_utils.vol_to_surf(shap_img, fsaverage.pial_left)
    nonzero = texture[np.abs(texture) > 0]
    vmax = np.percentile(np.abs(nonzero), 95) if len(nonzero) > 0 else 1.0
    thresh = np.percentile(np.abs(nonzero), 10) if len(nonzero) > 0 else 0.0

    out_path = FIGURES_DIR / f"shap_surf_{class_name}.png"

    surf_fig = plotting.plot_surf_stat_map(
        fsaverage.infl_left, stat_map=texture,
        hemi='left', view='lateral',
        threshold=thresh, colorbar=True,
        symmetric_cmap=None, vmax=vmax,
        bg_on_data=True, alpha=0.7,
        cbar_tick_format='%.2e'
    )
    surf_fig.savefig(str(out_path), dpi=150, bbox_inches='tight')
    plt.close(surf_fig)

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(plt.imread(out_path))
    ax.set_title(f"SHAP: {class_name}", fontsize=14, pad=10)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
TOP_PERCENTILE = 25
winner_map = np.argmax(shap_voxel, axis=0)
max_shap = np.amax(shap_voxel, axis=0)
threshold = np.percentile(max_shap, TOP_PERCENTILE)
winner_values = np.where(max_shap >= threshold, winner_map + 1, 0).astype(np.float64)
winner_img = masker.inverse_transform(winner_values.reshape(1, -1))

view = plotting.view_img_on_surf(
    winner_img,
    surf_mesh='fsaverage',
    title="SHAP Somatotopic Map (1=Middle_Finger, 2=Hand, 3=Forearm, 4=Arm)",
    symmetric_cmap=False,
    vmin=1, vmax=4,
    threshold=1,
    vol_to_surf_kwargs={"interpolation": "nearest_most_frequent"}
)
view

In [ ]:
np.save(SHAP_DIR / "shap_values_pca.npy", np.array(shap_values))
np.save(SHAP_DIR / "shap_voxel_maps.npy", shap_voxel)
print(f"Saved to {SHAP_DIR}")